The three generated features (alternative headline, keyword, and summary) are deterministic because the temperature is set to 0. The result can be seen on the "GPT Features (Stable)".

In [ ]:
# Import necessary packages
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
import os
import csv

# Loads the environment variables from a .env file
_ = load_dotenv(find_dotenv())
client = OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY')
)

def getContent(inputFile):
    getFile = [f for f in os.listdir(inputFile) if f.endswith('.txt')]
    contents = {}
    for file in getFile:
        try:
            with open(os.path.join(inputFile, file), 'r', encoding='utf-8', errors='replace') as f:
                lines = f.readlines()
                title = lines[0].replace("Title: ", "").strip()
                date = lines[1].replace("Date: ", "").strip()
                contentIndexStart = lines.index("Content:\n") + 1
                content = ''.join(lines[contentIndexStart:])
                contents[file] = (title, date, content.strip())
        except Exception as e:
            print(f"Error reading file {file}: {e}")
    return contents

def getResponses(content, promptlst):
    prompts = "\n".join([f"Prompt {i+1}: {prompt}" for i, prompt in enumerate(promptlst)])
    completion = client.chat.completions.create(
         model="gpt-4.1",
         messages=[
            {"role": "system", "content": "You are a helpful economist and journalist"},
            {"role": "user", "content": f"{prompts}\n\n{content}\n\n'Please just write the response; you do not need to write prompt 1, etc.'\n\n'Each response only needs to be divided by one newline'"}
        ],
        temperature= 0,
        top_p= 0,
        seed= 42
    )
    responses = completion.choices[0].message.content.split("\n\n")
    usedToken = completion.usage.total_tokens
    return responses, usedToken

def process(inputFile, outputFile):
    contents = getContent(inputFile)
    rows = []
    promptlst = [
        "Please come up with a high-quality (attractive, clear, accurate, and inoffensive) headline for the following news article, please only generate the headline without quotation marks",
        "Please identify five keywords for the following news article, and make each one or two words long (when generating the keywords, please use the format: ..., ..., ...)",
        "Generate a summary of the following news article using no more than two sentences. Keep the summary as brief as possible while including key information. Ensure the summary appears on a single line without any line breaks"
    ]

    for file, (title, date, content) in contents.items():
        responses, usedToken = getResponses(content, promptlst)
        rows.append([date, title] + responses + [usedToken])

    with open(outputFile, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Date", "Original Headline", "Generated Headline", "Keywords", "Generated Summary", "Tokens Used"])
        writer.writerows(rows)

inputFile = "decoy"
outputFile = "decoy"
process(inputFile, outputFile)